In [ ]:
# capstone project -- function 8 (8D), week N

import numpy as np
import matplotlib.pyplot as plt

from scipy.optimize import minimize
from scipy.spatial import ConvexHull
from scipy.stats import spearmanr

from bayes_tools import (
    fitting,
    normalize, initial_bounds, validate_bounds_consistency,
    generate_next_point, ucb_acquisition,
    append_observations, compute_iteration_diagnostics,
    fit_gp, get_length_scales, loo_predictions,
    compare_kappa_proposals,
    backtest_acquisitions, print_backtest_summary,
)
from viz_tools import (
    plot_nd_slices, plot_loo_calibration, plot_kappa_sensitivity,
    plot_acquisition_backtest,
    plot_convergence, plot_acquisition_decay, plot_uncertainty_shrinkage,
    plot_step_distance,
)

# Function 8

8D — the highest-dimensional function in the set. 43 observations (40 initial + weeks 2–4), all in domain and all used for modelling. Search domain is the unit cube `[0,1]^8`. **Acquisition is `ucb` with `kappa = 0.25`, searching the six live axes {x0, x1, x2, x3, x5, x6} with x4 and x7 held at the incumbent.**

**This week's proposal is not the acquisition's output.** It is a deliberate controlled experiment: **`[0.106878, 0.190311, 0.050599, 0.209113, 1.000000, 0.414903, 0.213780, 0.603135]`** — the incumbent with **x4 moved to 1.0 and the other seven coordinates unchanged**. Reasoning below. Set `USE_CONFOUND_TEST = False` in the proposal cell to fall back to the `kappa=0.25` proposal.

## Progress, and the y-value ambiguity is now resolved

| week | y | note |
|---|---|---|
| 2 | 9.927989 | new best at the time |
| 3 | 9.802053 | slightly below week 2 |
| 4 | **9.945552** | incumbent |

Earlier versions flagged a risk that week 2's `9.9279892899925` might have been `29.93`, and said the analysis would change substantially if so. **Settled: it was 9.93.** Weeks 3 and 4 came back at 9.802 and 9.946, so all three sit inside [9.80, 9.95]; a 29.93 would be wildly out of line with its own two successors.

## Why this week tests x4 instead of exploiting

Holding x4 and x7 has been the right call, but **the reason given for it was wrong**, and checking it exposed a confound worth a week to break.

The old reason was that upper-bound contact meant `pad_fraction` was choosing the coordinate. That is void — bounds are now the true domain, so an upper-bound contact is a legitimate "as high as allowed" answer.

Re-derived, the picture is this. Including x4 in the search claims a **30× larger predicted gain**:

| axes searched | pred. mean | gain vs incumbent | where x4 goes |
|---|---|---|---|
| live only (committed) | +9.9518 | **+0.0063** | held at 0.5869 |
| live + x4 | +10.1306 | **+0.1850** | **1.0** (domain edge) |
| live + x7 | +9.9555 | +0.0099 | — |

But that +0.185 is not credible, and the reason is a confound:

```
rank   src         y       x4       x7
   1   wk4    9.9456   0.5869   0.6031
   2   wk2    9.9280   0.5869   0.6031
   3   wk3    9.8021   0.5869   0.6031
   4  init    9.5985   0.4039   0.8931
```

**The three highest `y` values are the three collected points, and all three sit at exactly x4 = 0.586936** — the only observations at that value, frozen there since week 2's proposal. So the GP cannot separate x4's contribution from anything else those three points share. With x4's length-scale **pinned at the 10.0 ceiling** — the GP's own statement that it found no structure along that axis — it fits the smoothest thing consistent with the cluster, a linear ramp, and extrapolates it to the boundary.

The raw data points the other way:

| evidence | direction |
|---|---|
| GP posterior sweep | **increasing**, swing 0.5163, monotone |
| Spearman(x4, y), initial 40 only | **−0.120** |
| Spearman(x4, y), all 43 | −0.060 |
| initial batch, 5 highest x4 | y = 8.54, 8.16, **5.84**, 6.45, 6.89 |
| mean y, high-x4 half vs low-x4 half | **−0.190** |

So the GP claims x4 increases `y` while the length-scale says it detected nothing and the rank correlation is slightly *negative*. One evaluation resolves it.

### The test

Move x4 to 1.0 and change nothing else. One coordinate differs from the incumbent, so the comparison against the incumbent's measured 9.945552 is a clean one-factor experiment.

- **H1 (the GP is right):** y comes back near **+10.12**, above the incumbent — x4 genuinely helps, and it should be searched from now on.
- **H2 (the data is right):** y comes back **below 9.9456** — the ramp was a confound artefact, holding x4 is confirmed, and the +0.185 should never be chased again.

Either result breaks the confound: x4 stops being constant across the collected points, so the next refit can separate its effect. x4 = 1.0 is 0.013 beyond the largest x4 ever observed (0.9869) but inside the domain, and it is the GP's own argmax — which makes it the strongest available test of the GP's claim.

**x7 is left confounded this week.** It has the same frozen-value problem, but its claimed gain is +0.0099 against x4's +0.1850, so x4 is the one worth a week.

## Why `kappa = 0.25` is unchanged

By the predicted-gain criterion (see CLAUDE.md), it is the largest `kappa` whose predicted mean still beats the incumbent:

| kappa | pred. mean | gain | step | on a domain edge |
|---|---|---|---|---|
| 0 (= `exploit`) | +9.9531 | +0.0075 | 0.083 | none |
| **0.25** | +9.9518 | **+0.0063** | 0.097 | none |
| 0.5 | +9.9375 | −0.0081 | 0.182 | none |
| 1.0 | +9.9899 → +9.8989 | −0.0466 | 0.310 | x0, x2 |

Function 6 moved 0.5 → 0.25 on this criterion and function 7 moved 0.25 → 0; here it confirms the existing value. Same rule, three different answers.

## What 8D breaks

- **The convex-hull test is vacuous.** All **43** observations are hull vertices and the hull encloses 0.46% of the observed bounding box, so any proposal reads as outside it. `Delaunay` is skipped entirely; the per-axis and domain-edge checks carry the weight.
- **`compute_iteration_diagnostics` needs `domain_grid_n = 4`.** The default 40 would build a `40**8 = 6.6e12`-point grid (~419,000 GB). At 4 the `domain_mean_std` column is very coarse — comparable across iterations within this notebook only.

## Notes

- **EI and PI are degenerate here**, and this is the one function where that is true. Their proposals are *identical* at 5% and 100% of the `y` spread (movement 0.0000), both collapsing to a predicted +8.7095 against an incumbent of 9.95. `xi` is not a live dial. Contrast functions 6 and 7, where `xi` genuinely moves the proposal — it was worth checking per function rather than assuming.
- **No y-scaling.** `y` spans 5.592 to 9.946 so `xi=0.01` is 0.23% of the spread; `kappa` is dimensionless regardless.
- **The model is the best-behaved in the set.** LOO 40/43 inside their own 95% intervals, worst-predicted is an ordinary initial point at −2.2 sigma (not the incumbent, not the newest point). The GP and the rank correlations agree on which axes matter for six of eight axes — they disagree only on x4 and x7, which is exactly the confound above, and the axis cell flags that disagreement explicitly.
- `plot_2d_bo` doesn't apply at D=8 — the GP view is `plot_nd_slices`, eight panels, with x7 flat and x4 nearly linear.


In [2]:
X_initial = np.load("initial_data/function_8/initial_inputs.npy")
y_initial = np.load("initial_data/function_8/initial_outputs.npy")
n_initial = len(y_initial)
D = X_initial.shape[1]

assert D == 8, f"Expected an 8D problem, got {D}D input -- check the loaded file."

# Once, at the very start of the capstone:
new_X = np.empty((0, D))
new_y = np.empty((0,))

## Update this once per week

Include the new X and y values from the previous week, oldest first -- row order must be true chronological order, or `compute_iteration_diagnostics` at the bottom is meaningless.

One observation is recorded, and it is the new best. Provenance is clean here: all coordinates positive, none on a bound, all inside the range already sampled on their own axis -- no sign flip (contrast functions 4 and 7) and no rounding overshoot (contrast function 4).

**One thing to double-check**: `y` is recorded as `9.9279892899925`, matching `outputs.txt`. It was supplied as "2 9.9279892899925", which appears to be an editor artifact. If the intended value was `29.9279892899925`, change it below and re-run everything -- a value that far above the rest would dominate the kernel fit the way function 4's -215 does, and none of the conclusions in the header would survive unchanged.

In [ ]:
# Append last week's result BEFORE proposing this week's point, e.g.:
#
# new_X, new_y = append_observations(new_X, new_y, x_next, the_result_you_got)

new_X = np.array([
    # week 2
    [0.095333, 0.261314, 0.113603, 0.282728, 0.586936, 0.54168, 0.291424, 0.603135],
    # week 3 -- x1/x3/x6 landed exactly on the 0.0 floor.
    [0.035759, 0.0, 0.07531, 0.0, 0.586936, 0.318282, 0.0, 0.603135],
    [0.106878, 0.190311, 0.050599, 0.209113, 0.586936, 0.414903, 0.21378 , 0.603135],
])

new_y = np.array([
    9.9279892899925,    # from outputs.txt. See the note above if 29.93 was intended.
    9.8020526511435,
    9.9455521132595
])

print("observations collected so far:", len(new_y))
print("week 2 y: %.6f | week 3 y: %.6f (change %+.6f)"
      % (new_y[0], new_y[-1], new_y[-1] - new_y[0]))
print("best before week 3: %.6f | best overall now: %.6f"
      % (max(y_initial.max(), new_y[0]), max(y_initial.max(), new_y.max())))
print("-> week 3 is slightly BELOW week 2: three coordinates were driven to the")
print("   0.0 floor and the response did not improve, so that direction is spent")


observations collected so far: 2
week 2 y: 9.927989 | week 3 y: 9.802053 (change -0.125937)
best before week 3: 9.927989 | best overall now: 9.927989
-> week 3 is slightly BELOW week 2: three coordinates were driven to the
   0.0 floor and the response did not improve, so that direction is spent


## Build the full dataset (initial + everything collected so far)

In [ ]:
X, y = append_observations(X_initial, y_initial, new_X, new_y)

print("X_initial shape:", X_initial.shape, "| combined X shape:", X.shape)
print("y range: %.6f to %.6f (spread %.6f, std %.6f)"
      % (y.min(), y.max(), y.max() - y.min(), y.std()))

# X is known to never be negative -- lower_limit=0.0 is mandatory. Note this
# makes the LOWER bound a real domain constraint while the upper bounds are an
# artifact of pad_fraction; the checks below treat them differently.
bounds = initial_bounds(X_initial, pad_fraction=1.0, lower_limit=0.0,
                        upper_limit=1.0)
# Bounds ARE the domain, so an observation could legitimately sit outside them
# (earlier weeks proposed points above 1 before the domain was known). Validate
# the in-domain subset and report any others rather than dying on them. No
# observation is currently outside, so this is future-proofing.
_in_dom = np.all((X >= bounds[:, 0]) & (X <= bounds[:, 1]), axis=1)
validate_bounds_consistency(X[_in_dom], bounds)
if not _in_dom.all():
    print(f"\n{int((~_in_dom).sum())} observation(s) OUTSIDE the [0,1] domain:")
    for i in np.flatnonzero(~_in_dom):
        over = [f"x{d}={X[i, d]:.4f}" for d in range(D) if X[i, d] > 1.0]
        print(f"  {', '.join(over)}   y={y[i]:+.5g}")
    print("  Kept in the fit unless they distort it -- measure before excluding.")
print("\nBounds (upper):", np.round(bounds[:, 1], 4))

# The 0.0 floor is a real domain constraint, so a coordinate sitting exactly
# on it is legitimate -- but it also means the axis has run out of room in
# that direction, which is worth seeing per week.
print("\ncollected points vs the INITIAL batch's range:")
for r in range(len(new_X)):
    n_out = n_floor = 0
    for d in range(D):
        outside = (new_X[r, d] > X_initial[:, d].max()
                   or new_X[r, d] < X_initial[:, d].min())
        n_out += outside
        n_floor += new_X[r, d] == 0.0
        print(f"  week {r + 2}  x{d}={new_X[r, d]:.6f}  initial range"
              f" [{X_initial[:, d].min():.4f}, {X_initial[:, d].max():.4f}]"
              f"  outside={outside}{'  <- on the 0.0 floor' if new_X[r, d] == 0.0 else ''}")
    print(f"  -> week {r + 2}: {n_out} of {D} coordinates outside,"
          f" {n_floor} on the 0.0 floor, y={new_y[r]:.6f}")

# At D=8 the convex hull is vacuous: with this few points there is essentially
# no interior, so every proposal reads as outside it. Quantified rather than
# assumed. Delaunay is skipped -- expensive and uninformative at this D.
try:
    ch = ConvexHull(X)
    box_volume = float(np.prod(X.max(axis=0) - X.min(axis=0)))
    print(f"\nobserved box volume {box_volume:.4g} | convex hull {ch.volume:.4g}"
          f" ({ch.volume / box_volume:.2%} of it)")
    print(f"{len(ch.vertices)} of {len(X)} observations are hull vertices")
    if len(ch.vertices) == len(X):
        print("-> EVERY point is a vertex: the hull has no interior, so a")
        print("   hull-membership test carries no information at this D.")
except Exception as exc:
    print("\nconvex hull unavailable:", exc)

xi_frac = 0.01 / (y.max() - y.min())
print(f"\nxi=0.01 is {xi_frac:.4%} of the y spread"
      f" -> {'scaling NOT needed' if 0.001 <= xi_frac <= 0.1 else 'CONSIDER y-scaling'}")
print("(kappa is dimensionless, so UCB is unaffected either way)")

## Is the model stable and calibrated?

First, refit on the initial batch alone and predict the collected point -- a genuine out-of-sample test, since that fit never saw it. The miss in units of its own predicted sigma says how far the model's uncertainty can be trusted. This came back at 17 sigma on function 5 and changed the entire approach there; expect about 1.5 sigma here, the best in the set.

Second, compare kernels before and after. A single observation reshaping the model is a reason to distrust variance-driven acquisitions -- function 4's length-scales tripled from one point. Here the notable change is x3 coming *unpinned* (8.41 to 3.63), i.e. the new point made an axis look more relevant rather than less.

In [ ]:
with fitting("before/after fits for the new observation"):
    gp_before = fit_gp(X_initial, y_initial, bounds, n_restarts_optimizer=25, random_state=0)

with fitting("before/after fits for the new observation"):
    gp_check = fit_gp(X, y, bounds, n_restarts_optimizer=25, random_state=0)


# Fit is the initial batch only, so every collected row is out of sample. The
# model that actually proposed week 3 also had week 2, so this understates
# what was known at proposal time for later rows.
mu_b, sd_b = gp_before.predict(normalize(new_X, bounds), return_std=True)
print("out-of-sample test on the collected points (fit = initial batch only):")
for r in range(len(new_X)):
    miss = new_y[r] - mu_b[r]
    print(f"  week {r + 2}: predicted {mu_b[r]:+.6f} +/- {sd_b[r]:.6f}"
          f"   actual {new_y[r]:+.6f}")
    print(f"    miss {miss:+.6f} = {miss / sd_b[r]:+.1f} sigma"
          f"  -> {'well calibrated' if abs(miss / sd_b[r]) < 3 else 'POORLY calibrated'}")

ls_before, ls_after = get_length_scales(gp_before), get_length_scales(gp_check)
print("\nkernel before:", gp_before.kernel_)
print("kernel after :", gp_check.kernel_)
print("\nlength-scales before:", np.round(ls_before, 3))
print("length-scales after :", np.round(ls_after, 3))
print("ratio (after/before):", np.round(ls_after / ls_before, 3))

PINNED = 10.0  # fit_gp's default length_scale_bounds upper limit
print("\npinned (GP treats as irrelevant) before:",
      [d for d, v in enumerate(ls_before) if v >= 0.999 * PINNED] or "none")
print("pinned after                          :",
      [d for d, v in enumerate(ls_after) if v >= 0.999 * PINNED] or "none")

moved = np.max(np.abs(ls_after / ls_before - 1))
print(f"\nlargest relative length-scale change: {moved:.0%}")
if moved > 0.5:
    print("*** A length-scale moved by more than 50% from one observation. Check")
    print("    WHICH one: an axis becoming MORE relevant (length-scale shrinking)")
    print("    is benign; the dangerous case is an amplitude/length-scale blow-up")
    print("    that inflates far-field variance, as on function 4. ***")


## Which axes matter

Length-scales alone aren't sufficient: pinned at the 10.0 ceiling means "smooth, nearly linear over this domain", not "no effect" -- a mild monotone trend can still be present, and if it is, pushing along that axis is a real prediction rather than an optimiser artifact. That was function 5's trap.

So this cell measures GP mean-variation directly and pairs it with model-free rank correlations, then applies the flat-axis rule: flat if mean-variation is under 5% of the largest **or** the length-scale is at/above 9.0.

**This is the one function where the two diagnostics agree.** Expect x0, x2 and x6 to top both the mean-variation and the (absolute) rank-correlation ranking, and x4/x7 to come last on both. On function 5 they contradicted each other outright, which is why that notebook has to hedge; here they corroborate.

Note x4 qualifies as flat on the length-scale test (9.71) rather than the mean-variation test (0.86, which is 12% of x2's 7.30). Including it in the search is what pushed the unrestricted proposals onto x4's upper bound, so the length-scale criterion is doing real work.

In [ ]:
incumbent = X[np.argmax(y)]
print("incumbent (best observed):", np.round(incumbent, 6))
print("y = %.6f" % y.max())

print("\n%3s | %17s | %15s | %10s | %9s | %s"
      % ("ax", "GP mean-variation", "same, observed", "len-scale", "spearman", "rank agree?"))
spans = np.zeros(D)
srs = np.zeros(D)
for d in range(D):
    row = []
    for lo, hi in [(bounds[d, 0], bounds[d, 1]), (X[:, d].min(), X[:, d].max())]:
        grid = np.tile(incumbent, (300, 1))
        grid[:, d] = np.linspace(lo, hi, 300)
        m, _ = gp_check.predict(normalize(grid, bounds), return_std=True)
        row.append(m.max() - m.min())
    spans[d] = row[0]
    srs[d] = spearmanr(X[:, d], y)[0]

rank_span = np.argsort(np.argsort(-spans))
rank_sr = np.argsort(np.argsort(-np.abs(srs)))
for d in range(D):
    agree = "yes" if abs(int(rank_span[d]) - int(rank_sr[d])) <= 1 else "no"
    print("%3d | %17.4g | %15.4g | %10.3f | %+9.3f | %s (%d vs %d)"
          % (d, spans[d], 0.0 if d < 0 else spans[d], ls_after[d], srs[d],
             agree, rank_span[d] + 1, rank_sr[d] + 1))

FLAT_FRAC = 0.05
flat = [d for d in range(D)
        if spans[d] < FLAT_FRAC * spans.max() or ls_after[d] >= 0.9 * PINNED]
live = [d for d in range(D) if d not in flat]
print(f"\nflat axes (mean-variation < {FLAT_FRAC:.0%} of max, or length-scale >= {0.9 * PINNED}):",
      flat or "none")
for d in flat:
    why = []
    if spans[d] < FLAT_FRAC * spans.max():
        why.append("low mean-variation")
    if ls_after[d] >= 0.9 * PINNED:
        why.append(f"length-scale {ls_after[d]:.2f}")
    print(f"  x{d}: {', '.join(why)}")
print("axes searched:", live)

# A monotone axis whose best end is a domain bound will put the proposal ON that
# bound -- expected, not a defect. And an axis frozen at one value across every
# collected point cannot have its effect separated from those points' other
# features: the GP will happily fit a ramp through them. Flag both here.
for d in range(D):
    sr = spearmanr(X[:, d], y)[0]
    if abs(sr) > 0.4:
        which = "LOWER" if sr < 0 else "UPPER"
        print(f"  x{d} is monotone (spearman {sr:+.3f}): best end is the {which}"
              " domain bound, so a proposal sitting there is expected")

for d in range(D):
    vals = np.unique(np.round(new_X[:, d], 9))
    if len(new_X) > 1 and len(vals) == 1:
        print(f"  *** x{d} is FROZEN at {vals[0]:.6f} across all {len(new_X)} collected"
              " points.")
        print(f"      Its effect cannot be separated from whatever else those points"
              " share, so")
        print(f"      the GP's direction on x{d} (spearman {spearmanr(X[:, d], y)[0]:+.3f})"
              " is confounded. See the header. ***")
order = np.argsort(spans)[::-1]
print("dominance: x%d is only %.1fx the next (x%d) -> no single dominant axis"
      % (order[0], spans[order[0]] / spans[order[1]], order[1]))

## Backtest acquisition functions (using only data already collected)

Repeatedly splits the data into a "seed" set (fits the GP) and a held-out "candidate" set (true y known, hidden from the fit), then sees which config would have picked the best candidate most often. No new evaluations spent. `xi` is sized as 1% of the `y` spread.

With 41 points this is better powered than on the smaller functions -- but the structural caveat is unchanged: the metric scores recognition of points whose `y` is already known, which rewards ranking by posterior mean and gives no credit for reducing uncertainty, so it favours `exploit` and low `kappa` on every function regardless of what is appropriate. It ranked function 3's chosen config last.

Watch how little discrimination there is among the top seven: expect all of them at **median regret 0**, with the separation almost entirely in `max_variance`'s tail. Read it as agreeing with the choice rather than establishing it.

In [ ]:
xi_raw = 0.01 * (y.max() - y.min())
backtest_configs = [
    {"name": "ucb_k0.25", "acquisition": "ucb", "kappa": 0.25},
    {"name": "ucb_k0.5", "acquisition": "ucb", "kappa": 0.5},
    {"name": "ucb_k1",   "acquisition": "ucb", "kappa": 1.0},
    {"name": "ucb_k2",   "acquisition": "ucb", "kappa": 2.0},
    {"name": "ucb_k5",   "acquisition": "ucb", "kappa": 5.0},
    {"name": "ei",       "acquisition": "ei",  "xi": xi_raw},
    {"name": "pi",       "acquisition": "pi",  "xi": xi_raw},
    {"name": "exploit",  "acquisition": "exploit"},
    {"name": "max_var",  "acquisition": "max_variance"},
]
print(f"xi for the PI/EI rows: {xi_raw:.6f} (1% of the y spread)\n")

with fitting("acquisition backtest, 50 splits"):
    backtest_results = backtest_acquisitions(
        X, y, bounds, backtest_configs,
        n_repeats=50, seed_frac=0.5, maximize=True,
        gp_kwargs={"n_restarts_optimizer": 15}, random_state=0,
    )

print_backtest_summary(backtest_results)

# ---------------------------------------------------------------------------
# Which acquisition appears best on the evidence so far?
#
# Every config is scored on the SAME splits, so they are compared PAIRED: the
# per-split difference in regret is far less noisy than the two means
# separately, and its standard error says whether a gap is real. "Within noise"
# means |mean difference| <= 2 standard errors. Plain arithmetic, no test.
#
# Read it knowing the metric's bias (CLAUDE.md): it scores RECOGNITION of points
# whose y is already known, which is an exploitation task. It structurally
# favours `exploit` and low `kappa` and penalises `max_variance` on every
# function, so a low-kappa config at the top is close to tautological.
# ---------------------------------------------------------------------------
CHOSEN = "ucb_k0.25"        # must name the committed KAPPA; asserted where KAPPA is set

assert CHOSEN in backtest_results, (
    f"{CHOSEN!r} is not among the backtest configs, so the committed setting is "
    f"never scored. Add a row for it. Have: {sorted(backtest_results)}"
)

names = list(backtest_results)
mean_r = {k: float(backtest_results[k]["regret"].mean()) for k in names}
med_r = {k: float(np.median(backtest_results[k]["regret"])) for k in names}
ranked = sorted(names, key=lambda k: mean_r[k])
leader = ranked[0]
n_splits = len(backtest_results[leader]["regret"])


def paired_gap(a, b):
    """Mean per-split regret difference (b - a), and its standard error."""
    d = backtest_results[b]["regret"] - backtest_results[a]["regret"]
    return float(d.mean()), float(d.std(ddof=1) / np.sqrt(len(d)))


tied = [k for k in ranked[1:]
        if abs(paired_gap(leader, k)[0]) <= 2 * paired_gap(leader, k)[1]]
worse = [k for k in ranked[1:] if k not in tied]

print("\n=== which acquisition appears best in our tests so far? ===")
print(f"leader by mean regret   : {leader:>10}  ({mean_r[leader]:.4f})")
best_med = min(names, key=lambda k: med_r[k])
print(f"leader by median regret : {best_med:>10}  ({med_r[best_med]:.4f})"
      + ("   (agrees)" if best_med == leader else "   (DISAGREES with the mean)"))

print(f"\npaired against {leader}, over the same {n_splits} splits:")
for k in ranked[1:]:
    gap, se = paired_gap(leader, k)
    print(f"  {k:>10}: {gap:+.4f} +/- {se:.4f} ({gap / se:>5.1f} s.e.)"
          f"   {'within noise' if k in tied else 'clearly worse'}")

print(f"\nindistinguishable from the leader : {', '.join([leader] + tied)}")
print(f"clearly worse                     : {', '.join(worse) or 'none'}")

rank = ranked.index(CHOSEN) + 1
gap, se = paired_gap(leader, CHOSEN)
print(f"\nthis notebook proposes with {CHOSEN}: ranked {rank} of {len(ranked)}", end="")
if CHOSEN == leader:
    print(" -- it leads.")
else:
    print(f", {gap:+.4f} +/- {se:.4f} behind {leader}"
          f" ({'within noise' if CHOSEN in tied else 'a REAL gap'}).")
print("Do NOT switch on this table alone -- see the bias note above, and check")
print("where each candidate would actually propose before acting on it.")

plot_acquisition_backtest(backtest_results)
plt.show()


## Why not EI or PI: they are degenerate on this function

Worth demonstrating rather than asserting, because the backtest above ranks them respectably -- that is the recognition metric talking, not their actual proposals.

Run at any `xi`, EI and PI here propose a region with predicted mean around **+0.018** against an incumbent of 9.93. The mechanism is the one seen on function 4: the improvement term `mu - y_best - xi` is hopeless everywhere in this large 8D box, so EI collapses onto its `sigma*phi(z)` term and silently becomes a variance-seeker. Sweeping `xi` across two orders of magnitude changes nothing, which is the tell -- if `xi` is not a live dial, the acquisition is not doing what its name says.

UCB is the only family here whose exploration weight remains controllable.

In [ ]:
spread = y.max() - y.min()

# All five fits happen first, inside one fitting() block, and the rows are built
# but not printed. Printing inside the loop put each fit's ConvergenceWarnings
# between the table rows.
_rows = []
with fitting("EI/PI xi sweep, 5 fits"):
    for acq, xi in [("ei", 0.01 * spread), ("ei", 0.05 * spread), ("ei", 0.25 * spread),
                    ("pi", 0.01 * spread), ("pi", 0.25 * spread)]:
        xn, g = generate_next_point(X, y, bounds, acquisition=acq, xi=xi, maximize=True,
                                    n_restarts=40, random_state=0)
        m, sd = g.predict(normalize(xn.reshape(1, -1), bounds), return_std=True)
        edge = sum(np.isclose(xn[d], bounds[d, 0]) or np.isclose(xn[d], bounds[d, 1])
                   for d in range(D))
        out = sum(not (X[:, d].min() <= xn[d] <= X[:, d].max()) for d in range(D))
        _rows.append((f"{acq} xi={xi:.3f}", m[0], sd[0], edge, out))

print(f"{'config':>16} | {'pred mean':>10} | {'pred std':>9} | on bound | outside obs")
for name, m, sd, edge, out in _rows:
    print(f"{name:>16} | {m:+10.4f} | {sd:9.4f} | {edge:>3}/{D}   | {out}/{D}")
print(f"\nincumbent y = {y.max():.4f} -- these proposals are catastrophically worse,")
print("and xi has no effect across a 25x range, so it is not a live dial.")
print("EI/PI have degenerated into variance-seekers. Use UCB.")

## Compare kappa values -- the deciding cell

Two views. First `compare_kappa_proposals`, unrestricted over all eight axes, shown to demonstrate the corner problem: with `pad_fraction=1.0` doubling every axis, the box has around 256 corners and almost all its volume sits far from any data, so anything carrying a variance term runs outward. Expect coordinate-on-bound counts to climb from 3 of 8 at `kappa=0` to 8 of 8 by `kappa=2`.

Then the restricted search over the live axes only, holding x4 and x7 at the incumbent, which is what the choice is made from.

Bound contact is split into two categories because they mean different things. **Upper** bounds come from `pad_fraction` and are arbitrary -- a coordinate pinned there means the padding is choosing it rather than the model, and that is disqualifying. **Lower** bounds are all `0.0`, a real domain constraint, so a coordinate there is a genuine "as low as physically allowed" prediction -- and on x0, x2 and x6 that is corroborated by clearly negative rank correlations.

`kappa=0.25` is committed: it touches no upper bound, costs essentially nothing against pure exploitation (+10.155 vs +10.156), and keeps a small hedge. `kappa >= 1` starts pulling x0 to the floor as well.

In [ ]:
def bound_report(p):
    """Report which coordinates sit on a bound.

    NOTE the reading changed once bounds became the true domain [0,1]. Under the
    old padded bounds, upper contact meant pad_fraction was choosing the point.
    Now BOTH edges are real domain limits, so contact means the constrained
    optimum is on the boundary -- which on a monotone axis is exactly right.
    Neither edge is automatically a defect; judge it per axis.
    """
    upper = [d for d in range(D) if np.isclose(p[d], bounds[d, 1])]
    lower = [d for d in range(D) if np.isclose(p[d], bounds[d, 0])]
    return upper, lower


print("=== unrestricted over all 8 axes (shown to demonstrate the corner problem) ===")
with fitting("kappa sweep (one shared GP)"):
    kappa_rows = compare_kappa_proposals(X, y, bounds, kappa_values=[0.0, 0.5, 1.0, 2.0, 5.0],
                                          maximize=True, n_restarts=40, random_state=0)

print(f"{'kappa':>6} | {'pred mean':>10} | {'on bound':>9} | {'UPPER bound':>22}")
for row in kappa_rows:
    up, lo = bound_report(row["x_next"])
    print(f"{row['kappa']:6g} | {row['pred_mean']:+10.4f} | {len(up) + len(lo):>7}/{D}"
          f" | {str(up) if up else 'none':>22}")

plot_kappa_sensitivity(X, y, bounds, gp_check, kappa_rows, ucb_acquisition, maximize=True)
plt.show()


def restricted_ucb(kappa, n_starts=100):
    """Maximise UCB over the live axes only, holding flat axes at the incumbent."""
    lo = bounds[live, 0]
    hi = bounds[live, 1]

    def neg(v):
        p = incumbent.copy()
        p[live] = v
        return -ucb_acquisition(normalize(p.reshape(1, -1), bounds), gp_check,
                                 kappa=kappa, maximize=True)[0]

    best_val, best_v = np.inf, None
    rng = np.random.default_rng(0)
    for start in rng.uniform(lo, hi, size=(n_starts, len(live))):
        res = minimize(neg, start, method="L-BFGS-B", bounds=list(zip(lo, hi)))
        if res.fun < best_val:
            best_val, best_v = res.fun, res.x
    p = incumbent.copy()
    p[live] = best_v
    m, s = gp_check.predict(normalize(p.reshape(1, -1), bounds), return_std=True)
    return p, m[0], s[0]


print(f"\n=== restricted to the live axes x{live}, holding x{flat} at the incumbent ===")
print(f"{'kappa':>6} | {'pred mean':>10} | {'pred std':>9} | {'UPPER bnd':>12}"
      f" | {'lower bnd (ok)':>18} | {'dist':>7}")
for k in [0.0, 0.25, 0.5, 1.0]:
    p, m, s = restricted_ucb(k)
    up, lo_ = bound_report(p)
    print(f"{k:6g} | {m:+10.4f} | {s:9.4f} | {str(up) if up else 'none':>12}"
          f" | {str(lo_) if lo_ else 'none':>18} | {np.linalg.norm(p - incumbent):7.4f}")
print(f"incumbent y = {y.max():.6f}")

# Committed choice -- reused by the proposal, the slice plot, and the diagnostics
# replay below, so they can't silently drift apart.
KAPPA = 0.25
print(f"\nusing kappa = {KAPPA}, searching only x{live}")
print("chosen as the largest kappa touching no UPPER bound while keeping a hedge")

# The verdict cell above reports how the committed setting ranks, which only
# means something if CHOSEN names this KAPPA. Config names follow
# f"ucb_k{kappa:g}", so this is checkable rather than a comment to remember.
_expected = f"ucb_k{KAPPA:g}"
assert CHOSEN == _expected, (
    f"CHOSEN={CHOSEN!r} does not match the committed KAPPA={KAPPA}"
    f" (expected {_expected!r}). The backtest verdict above refers to a config"
    " this notebook does not use -- fix one or the other."
)


## Propose the next point

The live axes come from the restricted UCB search; x4 and x7 are held at the incumbent because the acquisition is effectively flat along them and letting it choose them pushed x4 onto its upper bound.

The convex-hull check is omitted here -- at D=8 with 41 points every observation is a hull vertex, so the test carries no information (see the dataset cell). The per-axis range check and the upper/lower bound split are what matter.

In [ ]:
USE_CONFOUND_TEST = True     # False -> take the kappa=0.25 acquisition proposal
CONFOUND_AXIS = 4            # the axis frozen across every collected point
CONFOUND_VALUE = 1.0         # the GP's own argmax on that axis; a domain edge

gp = gp_check
x_acq, mu_acq, sigma_acq = restricted_ucb(KAPPA, n_starts=200)

# The controlled test: the incumbent with ONE coordinate moved. Because only x4
# differs, comparing the result against the incumbent's MEASURED y is a clean
# one-factor experiment -- which is the whole point. See the header.
x_test = incumbent.copy()
x_test[CONFOUND_AXIS] = CONFOUND_VALUE
_m, _s = gp.predict(normalize(x_test.reshape(1, -1), bounds), return_std=True)

print("--- candidates ---")
for label, cand, m_, s_ in (("acquisition (kappa=%s)" % KAPPA, x_acq, mu_acq, sigma_acq),
                            ("confound test (x%d -> %.1f)" % (CONFOUND_AXIS, CONFOUND_VALUE),
                             x_test, _m[0], _s[0])):
    print(f"  {label:>28}: pred {m_:+.6f} +/- {s_:.4f}"
          f"  gain {m_ - y.max():+.6f}  step {np.linalg.norm(cand - incumbent):.4f}")

if USE_CONFOUND_TEST:
    x_next, mu_next, sigma_next = x_test, _m[0], _s[0]
    n_diff = int(np.sum(~np.isclose(x_next, incumbent)))
    print(f"\nCHOSEN: the confound test. It differs from the incumbent on exactly"
          f" {n_diff} coordinate(s),")
    print(f"so the result compared with the incumbent's measured {y.max():.6f} isolates"
          f" x{CONFOUND_AXIS}'s effect.")
    print(f"  H1 (GP right): y comes back near {_m[0]:+.2f}, ABOVE the incumbent"
          f" -> x{CONFOUND_AXIS} genuinely helps and should be searched.")
    print(f"  H2 (data right): y comes back BELOW {y.max():.4f} -> the ramp was a"
          f" confound artefact; keep holding x{CONFOUND_AXIS}.")
    print("Either way the confound is broken for next week's refit.")
else:
    x_next, mu_next, sigma_next = x_acq, mu_acq, sigma_acq
    print("\nCHOSEN: the acquisition proposal.")

print(f"\n--- Next point to evaluate (bounds shape {bounds.shape}) ---")
print("x_next:", np.round(x_next, 6))
print(gp.kernel_)
print(f"GP predicted mean: {mu_next:.6f}, predicted std: {sigma_next:.6f}")
print(f"current best observed y: {y.max():.6f}"
      f"  -> predicted improvement: {mu_next - y.max():+.6f}")
held = [d for d in flat if d != CONFOUND_AXIS] if USE_CONFOUND_TEST else flat
print(f"flat axes x{held} held at the incumbent:"
      f" {np.allclose(x_next[held], incumbent[held])}"
      + (f"  (x{CONFOUND_AXIS} is deliberately moved this week)" if USE_CONFOUND_TEST else ""))

# Bounds are the TRUE DOMAIN, so neither edge is automatically a defect. The
# question is whether the axis is monotone toward that edge (expected) or not.
up, lo = bound_report(x_next)
for label, dims, is_upper in (("UPPER", up, True), ("lower", lo, False)):
    print(f"\non the {label} domain edge:", dims or "none")
    for d in dims:
        sr = float(srs[d])
        if d == CONFOUND_AXIS and USE_CONFOUND_TEST:
            print(f"  x{d}: DELIBERATE -- this is the confound test, not the acquisition")
        elif abs(sr) > 0.4 and ((sr > 0) == is_upper):
            print(f"  x{d}: EXPECTED -- monotone (spearman {sr:+.3f}) with its best end here")
        else:
            print(f"  x{d}: QUESTION IT -- spearman {sr:+.3f} does not point this way")

outside = [d for d in range(D) if not (X[:, d].min() <= x_next[d] <= X[:, d].max())]
print("outside the observed range on its own axis:", outside or "none")

print(f"\ndistance from the incumbent: {np.linalg.norm(x_next - incumbent):.6f}")
print("nearest 3 observations to x_next:")
for i in np.argsort(np.linalg.norm(X - x_next, axis=1))[:3]:
    print(f"  dist={np.linalg.norm(X[i] - x_next):.4f}  y={y[i]:+.6f}")

## Visualise the GP and acquisition function via 1D slices

Eight panels, each holding the other seven dimensions fixed at the current best observed point and sweeping one dimension. Dotted line is the fixed centre, dashed red is the proposed `x_next`, green is UCB.

Expect **x7 to be a flat line and x4 nearly flat** -- their y-axis ranges are about 0.08 and 0.86 against x2's 7.3. That is why those two coordinates are pinned at the incumbent rather than optimised. x0, x2 and x6 should show the most structure, matching both the length-scales and the rank correlations.

This is a *partial* view: eight 1D slices through one point say nothing about interactions between dimensions, and at D=8 that omission covers most of the space. Treat it as a sanity check on the marginal behaviour, not a picture of the surface.

In [ ]:
plot_nd_slices(
    X, y, bounds, gp,
    acquisition_fn=ucb_acquisition,
    x_next=x_next,
    acq_kwargs={"kappa": KAPPA, "maximize": True},
)
plt.show()

## Sanity-check the surrogate model: leave-one-out calibration

Refits the GP once per observation, leaving it out, and predicts it from the rest. At D=8 this is essentially the only way to judge the surrogate -- the slice plots cover a vanishing fraction of the space.

Expect the best coverage in the set: around 40 of 41 points inside their own 95% interval, with the worst-predicted being an ordinary initial observation missed by a modest margin -- not the newly collected point (predicted to 1.5 sigma before it arrived) and not the incumbent. Contrast function 7, where the incumbent itself is a 13.5 sigma LOO miss.

In [ ]:
with fitting("leave-one-out calibration"):
    pred_mean, pred_std = loo_predictions(X, y, bounds, gp_kwargs={"n_restarts_optimizer": 20})

plot_loo_calibration(y, pred_mean, pred_std)
plt.show()

within = np.abs(y - pred_mean) <= 1.96 * pred_std
print(f"points inside their own 95% LOO interval: {within.sum()}/{len(y)}")
worst = int(np.argmax(np.abs(y - pred_mean)))
print(f"worst-predicted: index {worst} -> true {y[worst]:.6f},"
      f" predicted {pred_mean[worst]:.6f} (std {pred_std[worst]:.6f})"
      f" = {(y[worst] - pred_mean[worst]) / pred_std[worst]:+.1f} sigma")
src = "initial batch" if worst < n_initial else f"collected #{worst - n_initial + 1}"
print(f"  that point is from the {src}")
print("is it the incumbent?", worst == int(np.argmax(y)))


## Iteration diagnostics

`compute_iteration_diagnostics` replays the ordered `X`/`y` to reconstruct what the acquisition value, GP hyperparameters, and domain-wide uncertainty were at each past proposal -- no persisted log involved.

**`domain_grid_n` MUST be lowered drastically at D=8.** Its `domain_mean_std` field averages the posterior std over a `domain_grid_n ** D` grid, so the default `domain_grid_n=40` means `40**8 = 6.6e12` points -- about **419,000 GB**. Even `domain_grid_n=10` would be 1e8 points. The cell below sizes the grid so the point count stays near 200,000, which gives `domain_grid_n=4` here: just four samples per axis.

At that resolution `domain_mean_std` is a very coarse estimate of domain-average uncertainty. It remains usable for comparing one iteration against another *within this notebook*, because every iteration uses the same grid, but its absolute value means little and it is not comparable across notebooks.

Two further caveats. The setting is taken from `KAPPA` so it can't drift from the proposal -- but the recorded observation came from the old week-1 sweep's `pi`/`xi=0.01`, so its replayed acquisition value describes a decision never made that way; treat that column as meaningless until the history is UCB throughout. And it assumes row order is true chronological order.

`plot_bo_diagnostics` is skipped (it hard-codes a 2D scatter panel and this is 8D); the trend plots below are dimension-agnostic and gated on having a few completed iterations.

In [ ]:
# compute_iteration_diagnostics builds a domain_grid_n ** D grid. The default of
# 40 is 40**8 = 6.6e12 points (~419,000 GB) at D=8. Size it so the grid stays
# ~200k points whatever D is -- here that means just 4 samples per axis.
DOMAIN_GRID_N = max(3, int(200_000 ** (1.0 / D)))
print(f"domain_grid_n = {DOMAIN_GRID_N} -> {DOMAIN_GRID_N ** D:,} grid points"
      f" (the default 40 would be {40 ** D:,})")
print("domain_mean_std is therefore very coarse: comparable across iterations")
print("within this notebook, not across notebooks or grid sizes.")

# The old `if len(new_y) == 0` guard is gone: new_y is populated above, so that
# branch was unreachable. The count comes from the MODELLING set rather than
# len(new_y), so it stays correct if rows are ever excluded from the fit.
n_replayed = len(y) - n_initial

with fitting("iteration-diagnostics replay"):
    history = compute_iteration_diagnostics(X, y, bounds, n_initial=n_initial,
                                            acquisition="ucb", kappa=KAPPA,
                                            maximize=True,
                                            domain_grid_n=DOMAIN_GRID_N)

print("\nBest y so far:", np.nanmax(history["y"]))
print("\nCAVEAT: the history is MIXED -- week 2 came from the old week-1 template's")
print("settings, and compute_iteration_diagnostics applies ONE setting to the whole")
print("history, so its acq_value column is an artefact of the mismatch for that row.")
print("The GP-hyperparameter and domain_mean_std columns do not depend on the")
print("acquisition and are fine throughout.")
print("completed iterations in the modelling set:", n_replayed)

if n_replayed >= 3:
    for plot_fn in (plot_convergence, plot_acquisition_decay,
                    plot_uncertainty_shrinkage, plot_step_distance):
        plot_fn(history)
        plt.show()
else:
    print(f"\nOnly {n_replayed} replayed iteration(s) -- need at least 3 before the")
    print("trend plots say anything. Skipping them; the raw fields are below.")

## Raw diagnostic fields

In [ ]:
print("y (last 5):", np.round(history["y"][-5:], 5))
print("iteration (last 5):", history["iteration"][-5:])
obs = ~np.isnan(history["acq_value"])
print("\nfor the proposed (non-initial) points only:")
print("  acq_value      :", history["acq_value"][obs])
print("  pred_mean      :", history["pred_mean"][obs])
print("  actual y       :", history["y"][obs])
print("  length_scale   :", history["length_scale"][obs])
print("  domain_mean_std:", history["domain_mean_std"][obs])

In [15]:
# The proposal as a hyphen-separated string, for submission.
print("-".join(f"{v:.6f}" for v in x_next))

# Full precision as well. 6 dp is fine to submit, but paste THIS into next
# week's new_X: a 6-dp copy of function 4's proposal rounded 4e-7 outside its
# own upper bound and tripped validate_bounds_consistency.
print("\nfull precision (use for next week's new_X):")
print("-".join(repr(float(v)) for v in x_next))

0.106878-0.190311-0.050599-0.209113-0.586936-0.414903-0.213780-0.603135

full precision (use for next week's new_X):
0.10687827978978602-0.1903112166112948-0.05059946299221537-0.2091127599584623-0.586936-0.4149030380730468-0.21377956170241738-0.603135
